In [0]:
from datetime import datetime
from pyspark.sql import DataFrame
from functools import reduce
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType
from delta.tables import DeltaTable

In [0]:
# ============================================================
# CONFIGURATION
# ============================================================


# Environment
# -------------------------------

storage_account_name = "southbeanadlsdev"

# ADLS Containers
bronze_base = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
silver_base = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"
gold_base   = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"



# Databricks Widgets
# -------------------------------

# Production
# table_name = dbutils.widgets.get("table_name")

# trigger_time = dbutils.widgets.get("trigger_time")
date_suffix = trigger_date.strftime('year=%Y/month=%m/day=%d')

# Development
table_name = "customers"



# Metadata Database
# -------------------------------

meta_db_password = dbutils.secrets.get(
    scope="keyvault-scope",
    key="AzureMetaDbpwd"
)

jdbc_url = (
    "jdbc:sqlserver://coffeebeansqlserver.database.windows.net:1433;"
    "database=metadatadb;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

jdbc_properties = {
    "user": "sqladmin",
    "password": meta_db_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


# -------------------------------
# Common SCD Columns


SYSTEM_COLUMNS = [
    "EffectiveStartDate",
    "EffectiveEndDate",
    "IsCurrent"
]


In [0]:
%run ./transformations

In [0]:
# ============================================================
# READ TRANSFORMATION CONFIGURATION
# ============================================================

from pyspark.sql.functions import col

# Read TransformationConfig table
config_df = (
    spark.read
         .jdbc(
             url=jdbc_url,
             table="TransformationConfig",
             properties=jdbc_properties
         )
)

# Get configuration for current table
config = (
    config_df
    .filter(col("TableName") == table_name)
    .first()
)

if config is None:
    raise Exception(
        f"No TransformationConfig found for table '{table_name}'."
    )

print(f"✓ Loaded TransformationConfig for {table_name}")

✓ Loaded TransformationConfig for customers


In [0]:
# ============================================================
# BUILD RUNTIME CONFIGURATION
# ============================================================

def parse_config_list(value):
    """
    Converts a comma-separated configuration value into a list.

    Examples
    --------
    None            -> []
    ""              -> []
    "A,B,C"         -> ["A", "B", "C"]
    """

    if value is None:
        return []

    value = value.strip()

    if value == "":
        return []

    return [
        item.strip()
        for item in value.split(",")
        if item.strip()
    ]


runtime = {

    "TableName": table_name,
    
    "MergeStrategy": (
        config["MergeStrategy"].upper()
        if config["MergeStrategy"]
        else None
    ),

    "SCDType": config["SCDType"],

    "BusinessKey": config["BusinessKey"],

    "CompareColumns": parse_config_list(
        config["CompareColumns"]
    ),

    "RequiredColumns": parse_config_list(
        config["RequiredColumns"]
    ),

    "NonNegativeColumns": parse_config_list(
        config["NonNegativeColumns"]
    ),

    "SurrogateKeyColumn": config["SurrogateKeyColumn"],

    
    "BronzePath": f"{bronze_base}{table_name.lower()}/{date_suffix}/",

    #"BronzePath": f"{bronze_base}{table_name.lower()}",

    "SilverPath": f"{silver_base}{table_name.lower()}",

    "GoldPath": f"{gold_base}{table_name.lower()}",

    # --------------------------------------------------------
    # Unity Catalog Tables
    # --------------------------------------------------------

    "SilverTable": f"silver.{table_name.lower()}",

    "GoldTable": f"gold.{table_name.lower()}"

}


# ------------------------------------------------------------
# Validate mandatory configuration
# ------------------------------------------------------------

if runtime["MergeStrategy"] is None:
    raise Exception("MergeStrategy is missing.")

if runtime["SurrogateKeyColumn"] is None:
    raise Exception("SurrogateKeyColumn is missing.")


print("=" * 60)
print("Runtime Configuration")
print("=" * 60)

for key, value in runtime.items():
    print(f"{key:<22}: {value}")

print("=" * 60)

Runtime Configuration
TableName             : customers
MergeStrategy         : MERGE
SCDType               : 2
BusinessKey           : CustomerID
CompareColumns        : ['CustomerName', 'Phone', 'City', 'IsDeleted']
RequiredColumns       : ['CustomerID', 'CustomerName']
NonNegativeColumns    : []
SurrogateKeyColumn    : CustomerSK
BronzePath            : abfss://bronze@southbeanadlsdev.dfs.core.windows.net/customers/year=2026/month=06/day=30/
SilverPath            : abfss://silver@southbeanadlsdev.dfs.core.windows.net/customers
GoldPath              : abfss://gold@southbeanadlsdev.dfs.core.windows.net/customers
SilverTable           : silver.customers
GoldTable             : gold.customers


In [0]:
# ============================================================
# READ BRONZE
# ============================================================

print("=" * 60)
print(f"Reading Bronze : {runtime['TableName']}")
print("=" * 60)

bronze_df = (
    spark.read
         .format("parquet")
         .load(runtime["BronzePath"])
)

print(f"Bronze Path    : {runtime['BronzePath']}")
print(f"Bronze Records : {bronze_df.count()}")

# Optional (Development)
display(bronze_df.limit(10))

print()

Reading Bronze : customers
Bronze Path    : abfss://bronze@southbeanadlsdev.dfs.core.windows.net/customers/year=2026/month=06/day=30/
Bronze Records : 30


customerid,customername,phone,city,createddate,lastmodifieddate,isdeleted
1,Arjun Nair,9876500001,Bangalore,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false
2,Priya Sharma,9876500002,Chennai,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false
3,Vikram Singh,9876500003,Mumbai,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false
4,Ananya Iyer,9876500004,Hyderabad,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false
5,Rohit Kumar,9876500005,Delhi,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false
6,Sneha Menon,9876500006,Bangalore,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false
7,Karthik Raj,9876500007,Chennai,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false
8,Pooja Verma,9876500008,Mumbai,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false
9,Naveen Kumar,9876500009,Hyderabad,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false
10,Aishwarya Rao,9876500010,Delhi,2026-06-24T14:10:42.846Z,2026-06-24T14:10:42.846Z,false


In [0]:
# ============================================================
# COMMON PROCESSING
# ============================================================

# ------------------------------------------------------------
# 1. Remove Duplicates
# ------------------------------------------------------------

print("1. Removing Duplicates...")

bronze_df = remove_duplicates(bronze_df)

print()


# ------------------------------------------------------------
# 2. Data Quality
# ------------------------------------------------------------

print("2. Data Quality Checks...")

valid_df, invalid_df = apply_data_quality(
    bronze_df,
    runtime["RequiredColumns"],
    runtime["NonNegativeColumns"]
)

print()

# ------------------------------------------------------------
# 3. Read Silver
# ------------------------------------------------------------

print("3. Reading Silver...")


silver_df = (
    spark.table(runtime["SilverTable"])
         .filter(col("IsCurrent")== True)
)

print(f"Silver Records : {silver_df.count()}")

print()


# ------------------------------------------------------------
# 4. Detect Schema Drift
# ------------------------------------------------------------

print("4. Detecting Schema Drift...")

drift = detect_schema_drift(
    valid_df,
    silver_df,
    system_columns=[
        runtime["SurrogateKeyColumn"],
        *SYSTEM_COLUMNS
    ]
)

log_schema_drift(
    runtime["TableName"],
    drift
)

validate_schema_drift(drift)

print()


# ------------------------------------------------------------
# 5. Handle Schema Evolution
# ------------------------------------------------------------

print("5. Handling Schema Evolution...")

handle_schema_evolution(
    runtime["SilverTable"],
    drift,
    valid_df
)

print()


# ------------------------------------------------------------
# 6. Refresh Silver
# ------------------------------------------------------------

print("6. Refreshing Silver Schema...")

if drift[new_columns]:
    silver_df = (
    spark.table(runtime["SilverTable"])
         .filter(col("IsCurrent") == True)
)
print("Silver schema refreshed.")

print()

1. Removing Duplicates...
Rows before deduplication : 20
Rows after deduplication  : 20
Duplicates removed        : 0

2. Data Quality Checks...
Total rows   : 20
Valid rows   : 20
Invalid rows : 0

3. Reading Silver...
Silver Records : 20

4. Detecting Schema Drift...
Schema Drift Report : Products
New Columns     : []
Missing Columns : []
Type Changes
None

5. Handling Schema Evolution...
✓ No schema evolution required.

6. Refreshing Silver Schema...
Silver schema refreshed.



In [0]:
# ============================================================
# STRATEGY PROCESSING
# ============================================================

merge_strategy = runtime["MergeStrategy"]
scd_type = runtime["SCDType"]


# ------------------------------------------------------------
# APPEND
# ------------------------------------------------------------

if merge_strategy == "APPEND":

    print("APPEND Strategy")

    append_to_silver(
        valid_df,
        runtime["SilverTable"]
    )


# ------------------------------------------------------------
# FULL
# ------------------------------------------------------------

elif merge_strategy == "FULL":

    print("FULL Strategy")

    overwrite_silver(
        valid_df,
        runtime["SilverTable"]
    )


# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

elif merge_strategy == "MERGE":

    print("MERGE Strategy")

    # --------------------------------------------------------
    # SCD Type 2
    # --------------------------------------------------------

    if scd_type == 2:

        print("SCD Type 2")

        new_records_df, changed_records_df, unchanged_records_df = (
            detect_scd_changes(
                valid_df,
                silver_df,
                runtime["BusinessKey"],
                runtime["CompareColumns"]
            )
        )

        print(f"New Records       : {new_records_df.count()}")
        print(f"Changed Records   : {changed_records_df.count()}")
        print(f"Unchanged Records : {unchanged_records_df.count()}")

        business_key_type = (
            valid_df
            .select(runtime["BusinessKey"])
            .schema[0]
            .dataType
        )

        merge_source_df = prepare_merge_source(
            new_records_df,
            changed_records_df,
            runtime["BusinessKey"],
            business_key_type
        )

        if merge_source_df.limit(1).count() == 0:

            print("No new or changed records. MERGE skipped.")

        else:

            merge_to_silver(
                merge_source_df,
                runtime["SilverTable"],
                runtime["BusinessKey"]
            )

    else:

        raise NotImplementedError(
            f"SCD Type {scd_type} is not implemented."
        )


# ------------------------------------------------------------
# UNKNOWN STRATEGY
# ------------------------------------------------------------

else:

    raise Exception(
        f"Unsupported Merge Strategy : {merge_strategy}"
    )